# 21.12 — Fuzzy logic

Fuzzy logic replaces brittle yes-or-no membership with degrees of truth that combine smoothly and defuzzify into actions.

Fuzzy logic handles symbolic categories whose boundaries are gradual. Expert-system rules can fire partially, and defuzzification turns graded conclusions into one action.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt

random.seed(2112)


def triangle(x, left, center, right):
    if x <= left or x >= right:
        return 0.0
    if x == center:
        return 1.0
    if x < center:
        return (x - left) / (center - left)
    return (right - x) / (right - center)


def clip_membership(xs, membership, strength):
    return [min(strength, value) for value in membership]


def centroid(xs, membership):
    numerator = sum(x * value for x, value in zip(xs, membership))
    denominator = sum(membership)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def build_fuzzy_ladder():
    rungs = []
    rungs.append({
        "name": "D1 two-membership toy rule",
        "inputs": {"focus": 0.7, "motion": 0.4},
        "rules": [("min", ["focus", "motion"], "slow", 4.0)],
        "expected_action": 4.0,
    })
    rungs.append({
        "name": "D2 three temperature rules",
        "inputs": {"cold": 0.2, "warm": 0.8, "hot": 0.1},
        "rules": [("single", ["cold"], "heat", 7.0), ("single", ["warm"], "hold", 5.0), ("single", ["hot"], "cool", 3.0)],
        "expected_action": 5.0,
    })
    rungs.append({
        "name": "D3 overlapping conflicting memberships",
        "inputs": {"cold": 0.5, "warm": 0.6, "humid": 0.7},
        "rules": [("min", ["cold", "humid"], "dry_heat", 6.5), ("single", ["warm"], "hold", 5.0), ("max", ["warm", "humid"], "fan", 4.0)],
        "expected_action": 4.8,
    })
    rungs.append({
        "name": "D4 HVAC control examples",
        "inputs": {"too_low": 0.4, "comfortable": 0.7, "too_high": 0.2, "occupied": 0.9},
        "rules": [("min", ["too_low", "occupied"], "heat", 7.0), ("single", ["comfortable"], "hold", 5.0), ("min", ["too_high", "occupied"], "cool", 3.0)],
        "expected_action": 5.2,
    })
    rungs.append({
        "name": "D5 multi-input noisy boundary controller",
        "inputs": {"too_low": 0.55, "comfortable": 0.5, "too_high": 0.45, "occupied": 0.8, "energy_saver": 0.6, "rapid_change": 0.7},
        "rules": [("min", ["too_low", "occupied"], "heat", 7.0), ("min", ["too_high", "occupied"], "cool", 3.0), ("single", ["comfortable"], "hold", 5.0), ("min", ["energy_saver", "rapid_change"], "soft", 4.5)],
        "expected_action": 5.0,
    })
    return rungs


## The concept, built once (D1)

A fuzzy set has membership $\mu(x)\in[0,1]$. The lesson operations are $\min(0.7,0.4)=0.4$, $\max(0.7,0.4)=0.7$, clipping a consequent at $0.6$, and a symmetric triangular output centered at $6$ with centroid $6.00$.

In [ ]:

lesson_and = min(0.7, 0.4)
lesson_or = max(0.7, 0.4)
lesson_xs = [4, 5, 6, 7, 8]
lesson_membership = [triangle(x, 4, 6, 8) for x in lesson_xs]
lesson_clipped = clip_membership(lesson_xs, lesson_membership, 0.6)
lesson_centroid = centroid(lesson_xs, lesson_membership)
lesson_smooth_low = 5 + 2 * math.tanh(-2)
lesson_smooth_high = 5 + 2 * math.tanh(2)

assert lesson_and == 0.4
assert lesson_or == 0.7
assert max(lesson_clipped) == 0.6
assert round(lesson_centroid, 2) == 6.00
assert round(lesson_smooth_low, 2) == 3.07
assert round(lesson_smooth_high, 2) == 6.93

print("AND", lesson_and)
print("OR", lesson_or)
print("centroid", lesson_centroid)
print("smooth range", round(lesson_smooth_low, 2), round(lesson_smooth_high, 2))


The controller combines rule strengths and defuzzifies action centers. This is compatibility with concepts like warm or high risk, not a probability frequency.

In [ ]:

def fuzzy_controller(inputs, rules):
    fired = []
    numerator = 0.0
    denominator = 0.0
    for connective, names, label, center in rules:
        values = [inputs[name] for name in names]
        if connective == "min":
            strength = min(values)
        elif connective == "max":
            strength = max(values)
        elif connective == "single":
            strength = values[0]
        else:
            raise ValueError(connective)
        fired.append({
            "label": label,
            "strength": strength,
            "center": center,
            "inputs": names,
        })
        numerator += strength * center
        denominator += strength
    action = numerator / denominator if denominator else 0.0
    return action, fired

lesson_inputs = {"focus": 0.7, "motion": 0.4}
lesson_rules = [("min", ["focus", "motion"], "slow", 6.0)]
lesson_action, lesson_fired = fuzzy_controller(lesson_inputs, lesson_rules)

assert lesson_fired[0]["strength"] == 0.4
print("rule strength", lesson_fired[0]["strength"])
print("action", lesson_action)


## The dataset ladder

D1–D5 add overlapping memberships, conflicting actions, HVAC examples, and noisy multi-input boundary cases.

In [ ]:

fuzzy_rungs = build_fuzzy_ladder()

for rung in fuzzy_rungs:
    print(rung["name"])
    print("  inputs", len(rung["inputs"]))
    print("  rules", len(rung["rules"]))
    print("  sample", rung["rules"][:2])


## Run the same method across D1–D5

In [ ]:

fuzzy_results = []

for rung in fuzzy_rungs:
    action, fired = fuzzy_controller(rung["inputs"], rung["rules"])
    error = abs(action - rung["expected_action"])
    fuzzy_results.append({
        "name": rung["name"],
        "inputs": len(rung["inputs"]),
        "rules": len(rung["rules"]),
        "action": action,
        "error": error,
        "fired": fired,
    })

print("rung | action | action error | rules fired")
for item in fuzzy_results:
    print(item["name"], round(item["action"], 3), round(item["error"], 3), item["rules"])


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, item in enumerate(fuzzy_results):
    ax = axes[0][index]
    labels = [entry["label"] for entry in item["fired"]]
    strengths = [entry["strength"] for entry in item["fired"]]
    ax.bar(labels, strengths)
    ax.set_ylim(0, 1)
    ax.set_title("D" + str(index + 1) + " activations")
    ax.tick_params(axis="x", rotation=45)

x_values = list(range(1, 6))
actions = [item["action"] for item in fuzzy_results]
errors = [item["error"] for item in fuzzy_results]
axes[1][0].plot(x_values, actions, marker="o", label="defuzzified action")
axes[1][0].plot(x_values, errors, marker="s", label="action error")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("value")
axes[1][0].set_title("smooth action/error vs complexity")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: confusing fuzzy membership with probability. Membership 0.55 means compatibility with too_low, not that 55% of rooms are too low.

In [ ]:

d5 = fuzzy_rungs[-1]
wrong_probability_language = d5["inputs"]["too_low"] * 100
fixed_action, fixed_fired = fuzzy_controller(d5["inputs"], d5["rules"])

print("wrong wording", str(round(wrong_probability_language)) + "% probability of too_low")
print("fixed wording", "compatibility degree", d5["inputs"]["too_low"])
print("rule trace")
for entry in fixed_fired:
    print(entry)
print("defuzzified action", round(fixed_action, 3))


## Evaluate it + Practice

- Metric: rule-output correctness, defuzzified action error, rules fired.
- No-skill baseline: hard threshold the largest label only.
- Cheap sanity check: lesson min/max/clip/centroid assertions pass.
- Ablation: replace min/max with crisp thresholds and smoothness drops.
- Failure signals: degrees are described as probabilities instead of memberships.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.